# 2. NumPy — Study Notes

Each subtopic: **core concept** (1-2 points) → **mini program (OOP)** → **output table** showing how the data is transformed.
Running example: pricing / revenue-operations data (products, regions, quarters, discounts).

| Section | Topic |
|---|---|
| 2.1 | Core Concepts |
| 2.2 | Array Creation |
| 2.3 | Array Manipulation |
| 2.4 | Mathematical Operations |
| 2.5 | Linear Algebra |
| 2.6 | Random Simulation |

In [4]:
import time
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
np.set_printoptions(precision=3, suppress=True)
print("NumPy", np.__version__)

NumPy 2.5.3


---
## 2.1 · Core Concepts

### ndarray, dimensions, shape, size, data types
- `ndarray` is a fixed-type, N-dimensional grid stored in one contiguous memory block; `ndim`, `shape`, `size` describe its geometry.
- `dtype` fixes bytes per element (`itemsize`), so `nbytes = size × itemsize`.

In [5]:
class ArrayInspector:
    """Profiles the geometry and memory footprint of named arrays."""

    def __init__(self, arrays: dict):
        self.arrays = arrays

    def profile(self) -> pd.DataFrame:
        rows = []
        for name, a in self.arrays.items():
            rows.append({
                "name": name, "ndim": a.ndim, "shape": a.shape, "size": a.size,
                "dtype": str(a.dtype), "itemsize": a.itemsize, "nbytes": a.nbytes,
            })
        return pd.DataFrame(rows).set_index("name")


arrays = {
    "unit_prices":   np.array([49.0, 79.0, 129.0]),
    "qty_by_region": np.array([[10, 20, 30], [5, 15, 25]]),
    "sales_cube":    np.arange(24).reshape(2, 3, 4),
    "prices_f32":    np.array([49.0, 79.0, 129.0], dtype=np.float32),
    "active_flags":  np.array([True, False, True]),
    "tiers":         np.array(["basic", "pro", "enterprise"]),
}
ArrayInspector(arrays).profile()

,ndim,shape,size,dtype,itemsize,nbytes
name,,,,,,
unit_prices,1,"(3,)",3,float64,8,24
qty_by_region,2,"(2, 3)",6,int64,8,48
sales_cube,3,"(2, 3, 4)",24,int64,8,192
prices_f32,1,"(3,)",3,float32,4,12
active_flags,1,"(3,)",3,bool,1,3
tiers,1,"(3,)",3,<U10,40,120


In [5]:
class DtypeLab:
    """Shows upcasting, explicit casting and integer wraparound."""

    @staticmethod
    def upcast(values):
        return np.array(values)

    @staticmethod
    def cast_table(x, targets):
        rows = []
        for t in targets:
            y = x.astype(t)
            rows.append({"target_dtype": t, "result": y.tolist(), "same_memory_as_x": np.shares_memory(x, y)})
        return pd.DataFrame(rows).set_index("target_dtype")


mixed = DtypeLab.upcast([1, 2.5, True])
print("mixed list [1, 2.5, True] ->", mixed, mixed.dtype)

revenue = np.array([120.7, 300.2, -45.9, 999.99])
DtypeLab.cast_table(revenue, ["int64", "float32", "int8", "uint8", "bool"])

NameError: name 'np' is not defined

In [4]:
class AxisAggregator:
    """Sums a labelled N-D array over each axis and reports the surviving axes."""

    def __init__(self, data, axis_names):
        self.data, self.axis_names = data, axis_names

    def sum_over(self, axis):
        out = self.data.sum(axis=axis)
        remaining = [n for i, n in enumerate(self.axis_names) if i != axis]
        return out, remaining

    def report(self):
        rows = []
        for ax, name in enumerate(self.axis_names):
            out, rem = self.sum_over(ax)
            rows.append({"collapse": f"axis={ax} ({name})", "result_shape": out.shape,
                         "remaining_axes": rem, "values": out.tolist()})
        return pd.DataFrame(rows).set_index("collapse")


# (quarter=2, region=3, product=2)
sales = np.array([[[100, 120], [80, 90], [60, 70]],
                  [[110, 130], [85, 95], [65, 75]]])
agg = AxisAggregator(sales, ["ac", "region", "product"])
display(agg.report())

# multi-axis reduction: revenue per region, collapsing quarter AND product
pd.Series(sales.sum(axis=(0, 2)), index=["North", "South", "West"], name="revenue_by_region")

NameError: name 'np' is not defined

### Memory layout
- Arrays are 1-D buffers plus `strides` (bytes to step per axis); C-order = row-major, F-order = column-major.
- Transpose and slice return **views** (new strides, same buffer) — no data is copied, but a view can be non-contiguous.

In [3]:
class LayoutReport:
    """Describes contiguity, strides and ownership of an array."""

    def __init__(self, a):
        self.a = a

    def describe(self):
        f = self.a.flags
        order = "C" if f["C_CONTIGUOUS"] else ("F" if f["F_CONTIGUOUS"] else "neither")
        return {"shape": self.a.shape, "strides": self.a.strides, "contiguous_order": order,
                "owns_data": f["OWNDATA"]}


base = np.arange(12, dtype=np.int64).reshape(3, 4)
variants = {
    "base (C-order)":      base,
    "asfortranarray":      np.asfortranarray(base),
    "base.T (view)":       base.T,
    "base[:, ::2] (view)": base[:, ::2],
    "slice .copy()":       base[:, ::2].copy(),
}
pd.DataFrame({k: LayoutReport(v).describe() for k, v in variants.items()}).T

NameError: name 'np' is not defined

---
## 2.2 · Array Creation

### np.array, zeros, ones, empty, full, arange, linspace, eye, random
- Constructors fix `shape` and `dtype` up front; `zeros/ones/full` initialise, `empty` does **not** (arbitrary leftover memory).
- `arange(start, stop, step)` is step-driven (stop excluded); `linspace(start, stop, num)` is count-driven (stop included).

In [2]:
class ArrayFactory:
    """Registry of every constructor, summarised in one table."""

    def __init__(self, seed=7):
        self.rng = np.random.default_rng(seed)

    def build(self):
        return {
            "np.array([[1,2,3],[4,5,6]])": np.array([[1, 2, 3], [4, 5, 6]]),
            "np.zeros((2,3))":             np.zeros((2, 3)),
            "np.ones((2,3), dtype=int)":   np.ones((2, 3), dtype=int),
            "np.empty((2,3))":             np.empty((2, 3)),
            "np.full((2,3), 99.5)":        np.full((2, 3), 99.5),
            "np.arange(0, 10, 3)":         np.arange(0, 10, 3),
            "np.linspace(0, 1, 5)":        np.linspace(0, 1, 5),
            "np.eye(3)":                   np.eye(3),
            "rng.random((2,3))":           self.rng.random((2, 3)),
        }

    def summary(self):
        rows = []
        for label, a in self.build().items():
            preview = "(uninitialised)" if "empty" in label else a.round(3).ravel().tolist()
            rows.append({"constructor": label, "shape": a.shape, "dtype": str(a.dtype), "values": preview})
        return pd.DataFrame(rows).set_index("constructor")


ArrayFactory().summary()

NameError: name 'np' is not defined

### Putting constructors to work
- `linspace` for continuous parameter sweeps, `arange` for discrete steps, `full` for constants, `eye(k)[labels]` for one-hot encoding.
- Compose constructors with broadcasting to build scenario tables in a few lines.

In [6]:
class ScenarioBuilder:
    """Builds a margin scenario cube and one-hot segment features from constructors."""

    def __init__(self, seed=21):
        self.rng = np.random.default_rng(seed)
        self.prices = np.arange(50, 251, 50)                 # 50..250 step 50
        self.discounts = np.linspace(0, 0.30, 4)             # 0, .1, .2, .3
        self.unit_cost = np.full(self.prices.shape, 35.0)    # constant cost

    def margin_table(self):
        net = self.prices[:, None] * (1 - self.discounts)    # (5,1)*(4,) -> (5,4)
        margin = (net - self.unit_cost[:, None]) / net
        return pd.DataFrame(margin.round(3), index=[f"${p}" for p in self.prices],
                            columns=[f"{d:.0%} off" for d in self.discounts])

    def one_hot_segments(self, n=6):
        labels = self.rng.integers(0, 3, size=n)             # 0=SMB 1=Mid 2=Ent
        onehot = np.eye(3, dtype=int)[labels]
        df = pd.DataFrame(onehot, columns=["SMB", "Mid", "Ent"])
        df.insert(0, "label", labels)
        return df


sb = ScenarioBuilder()
display(sb.margin_table())
sb.one_hot_segments()

NameError: name 'np' is not defined